<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 8: Film Oneri Sistemi

**MAKİNE ÖĞRENMESİ UZMANLIĞI** · Modül 8 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta08/hafta08_film_oneri_sistemi.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta08/hafta08_film_oneri_sistemi.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 8 - Film Öneri Sistemi

Bu derste:
- Kullanıcı-film puan matrisi oluşturacağız
- İçerik tabanlı filtreleme (Content-Based Filtering) uygulayacağız
- İşbirlikçi filtreleme (Collaborative Filtering) kavramını öğreneceğiz
- Kosinüs benzerliği ile kullanıcı bazlı öneriler yapacağız
- "Bu filmi izleyen şunu da izledi" mantığını kodlayacağız

## 1. Kütüphanelerin Yüklenmesi

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `seaborn` | İstatistiksel görselleştirme (Matplotlib üzerine kurulu) |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Gerçek MovieLens Verisini Yükleme

**MovieLens Small Dataset** — GroupLens tarafından toplanan gerçek film değerlendirmeleri. 100.000+ değerlendirme, 600+ kullanıcı, 9.000+ film.

**Kaynak:** [MovieLens](https://grouplens.org/datasets/movielens/) | [Kaggle](https://www.kaggle.com/datasets/grouplens/movielens-20m-dataset)

In [ ]:
# MovieLens Small Dataset (gerçek veri)
import zipfile, io, urllib.request

print("MovieLens veri seti indiriliyor...")
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
resp = urllib.request.urlopen(url)
z = zipfile.ZipFile(io.BytesIO(resp.read()))

ratings = pd.read_csv(z.open('ml-latest-small/ratings.csv'))
movies = pd.read_csv(z.open('ml-latest-small/movies.csv'))

# Birleştir
df = ratings.merge(movies, on='movieId')

print(f"Toplam değerlendirme: {len(df):,}")
print(f"Benzersiz kullanıcı: {df['userId'].nunique()}")
print(f"Benzersiz film: {df['movieId'].nunique()}")
print(f"\nPuan dağılımı:")
print(df['rating'].value_counts().sort_index())
print(f"\nÖrnek veriler:")
df.head(10)

### 2.2. Kullanıcı-Film Puan Matrisi

### Öneri Sistemi

Kullanıcı-ürün matrisi ve benzerlik hesabı ile öneri sistemi kuruyoruz. Cosine benzerliği, iki vektör arasındaki açıya dayalı bir benzerlik ölçüsüdür.

In [ ]:
# Kullanıcı-Film puan matrisi oluştur (pivot table)
# En çok değerlendirilen 200 filmi seçelim (performans için)
popular_movies = df['movieId'].value_counts().head(200).index
df_popular = df[df['movieId'].isin(popular_movies)]

# Pivot tablo: satırlar=kullanıcı, sütunlar=film, değerler=puan
user_movie_matrix = df_popular.pivot_table(
    index='userId', 
    columns='title', 
    values='rating'
)

print(f"Kullanıcı-Film Matrisi: {user_movie_matrix.shape}")
print(f"Dolumluk oranı: {user_movie_matrix.notna().sum().sum() / user_movie_matrix.size:.2%}")
print(f"\nİlk 5 kullanıcı, 5 film:")
user_movie_matrix.iloc[:5, :5]

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Puan dağılımı
all_ratings = ratings_df.values.flatten()
all_ratings = all_ratings[~np.isnan(all_ratings)]

plt.figure(figsize=(8, 5))
plt.hist(all_ratings, bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5], 
         color='steelblue', edgecolor='white', rwidth=0.8)
plt.xlabel('Puan')
plt.ylabel('Frekans')
plt.title('Puan Dağılımı')
plt.xticks([1, 2, 3, 4, 5])
plt.show()

## 3. İçerik Tabanlı Filtreleme (Content-Based Filtering)

Filmlerin tür özelliklerine göre benzerlik hesaplıyoruz. Eğer bir kullanıcı bir filmi beğendiyse, benzer türdeki filmler önerilir.

In [ ]:
# Filmler arası kosinüs benzerliği (tür vektörleri kullanarak)
film_similarity = cosine_similarity(genre_matrix)
film_sim_df = pd.DataFrame(film_similarity, index=film_names, columns=film_names)

# Isı haritası (ilk 15 film)
plt.figure(figsize=(14, 10))
sns.heatmap(film_sim_df.iloc[:15, :15], annot=True, fmt='.2f', 
            cmap='YlOrRd', square=True, linewidths=0.5)
plt.title('Film Benzerlik Matrisi (İçerik Tabanlı)', fontsize=14)
plt.tight_layout()
plt.show()

### Veri Gruplama ve Analiz

Verileri belirli kategorilere göre grupladıp özet istatistikler hesaplıyoruz. `groupby()` ile SQL'deki GROUP BY benzeri operasyonlar yapıyoruz.

In [ ]:
def content_based_recommend(film_adi, n=5):
    """Verilen filme benzer filmleri önerir (içerik tabanlı)."""
    if film_adi not in film_sim_df.index:
        print(f"'{film_adi}' bulunamadı!")
        return None
    
    similarities = film_sim_df[film_adi].drop(film_adi).sort_values(ascending=False)
    top_n = similarities.head(n)
    
    print(f"\n'{film_adi}' filmine benzer filmler:")
    print("=" * 50)
    for film, score in top_n.items():
        turler = films_df.loc[film]
        tur_listesi = ', '.join(turler[turler == 1].index)
        print(f"  {film} (Benzerlik: {score:.2f}) - Türler: {tur_listesi}")
    
    return top_n

content_based_recommend('Matrix', n=5)

In [ ]:
content_based_recommend('Titanik', n=5)

## 4. İşbirlikçi Filtreleme (Collaborative Filtering)

### Kavram Açıklaması

İşbirlikçi filtreleme iki ana yaklaşım içerir:

1. **Kullanıcı Tabanlı (User-Based)**: Benzer zevklere sahip kullanıcılar bulunur. "Senin gibi kullanıcılar şunu da beğendi."

2. **Öğe Tabanlı (Item-Based)**: Benzer puanlama örüntüsüne sahip filmler bulunur. "Bu filmi beğenenler şunu da beğendi."

Biz burada **Kullanıcı Tabanlı** yaklaşımı uygulayacağız.

In [ ]:
# NaN değerleri 0 ile doldur (benzerlik hesabı için)
ratings_filled = ratings_df.fillna(0)

# Kullanıcılar arası kosinüs benzerliği
user_similarity = cosine_similarity(ratings_filled)
user_sim_df = pd.DataFrame(user_similarity, index=user_names, columns=user_names)

plt.figure(figsize=(12, 10))
sns.heatmap(user_sim_df, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=0.5, vmin=0, vmax=1)
plt.title('Kullanıcı Benzerlik Matrisi', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Kullanıcı Tabanlı Öneri Sistemi

### Veri Gruplama ve Analiz

Verileri belirli kategorilere göre grupladıp özet istatistikler hesaplıyoruz. `groupby()` ile SQL'deki GROUP BY benzeri operasyonlar yapıyoruz.

In [ ]:
def user_based_recommend(user, n=5):
    """Kullanıcı tabanlı işbirlikçi filtreleme ile film önerir."""
    if user not in ratings_df.index:
        print(f"'{user}' bulunamadı!")
        return None
    
    # Kullanıcının izlemediği filmler
    unwatched = ratings_df.loc[user][ratings_df.loc[user].isna()].index
    
    if len(unwatched) == 0:
        print(f"{user} tüm filmleri izlemiş!")
        return None
    
    # Benzer kullanıcıları bul
    user_similarities = user_sim_df[user].drop(user).sort_values(ascending=False)
    top_similar_users = user_similarities.head(5).index
    
    # Benzer kullanıcıların puanlarını ağırlıklı ortalama ile hesapla
    predicted_scores = {}
    for film in unwatched:
        weighted_sum = 0
        similarity_sum = 0
        for sim_user in top_similar_users:
            rating = ratings_df.loc[sim_user, film]
            if not np.isnan(rating):
                sim = user_sim_df.loc[user, sim_user]
                weighted_sum += sim * rating
                similarity_sum += sim
        if similarity_sum > 0:
            predicted_scores[film] = weighted_sum / similarity_sum
    
    # En yüksek tahmini puanlı filmleri sırala
    recommendations = pd.Series(predicted_scores).sort_values(ascending=False).head(n)
    
    print(f"\n{user} için önerilen filmler:")
    print("=" * 50)
    for film, score in recommendations.items():
        print(f"  {film} (Tahmini Puan: {score:.2f})")
    
    return recommendations

user_based_recommend('Kullanıcı_1', n=5)

In [ ]:
user_based_recommend('Kullanıcı_5', n=5)

## 6. "Bu Filmi İzleyen Şunu da İzledi" Mantığı

### Veri Ön İşleme

Modeli eğitmeden önce veriyi temizliyoruz ve dönüştürüyoruz. Eksik değerleri doldurma, kategorik değişkenleri sayıya çevirme ve ölçeklendirme bu adımın parçasıdır.

In [ ]:
def also_watched(film_adi, n=5):
    """Verilen filmi izleyen kullanıcıların en çok izlediği diğer filmleri bulur."""
    if film_adi not in ratings_df.columns:
        print(f"'{film_adi}' bulunamadı!")
        return None
    
    # Bu filmi izleyen kullanıcılar
    watchers = ratings_df[film_adi].dropna().index
    
    if len(watchers) == 0:
        print(f"'{film_adi}' filmini kimse izlememiş!")
        return None
    
    # Bu kullanıcıların izlediği diğer filmler ve ortalama puanları
    other_films = ratings_df.loc[watchers].drop(columns=[film_adi])
    
    # İzlenme sayısı ve ortalama puan
    watch_count = other_films.notna().sum()
    avg_rating = other_films.mean()
    
    # Skor: izlenme sayısı * ortalama puan
    score = (watch_count * avg_rating).sort_values(ascending=False).head(n)
    
    print(f"\n'{film_adi}' filmini izleyenler şunları da izledi:")
    print("=" * 55)
    for film, s in score.items():
        count = watch_count[film]
        avg = avg_rating[film]
        if not np.isnan(avg):
            print(f"  {film} ({count} kişi izledi, Ort. Puan: {avg:.1f})")
    
    return score

also_watched('Matrix', n=5)

In [ ]:
also_watched('Başlangıç', n=5)

## 7. Basit Değerlendirme

Bilinen bir puanı gizleyip tahmin edebilir miyiz? Leave-one-out yöntemiyle test ediyoruz.

In [ ]:
# Basit değerlendirme: Bilinen puanları tahmin et
errors = []

for user in user_names[:5]:  # İlk 5 kullanıcı için test
    watched = ratings_df.loc[user].dropna()
    if len(watched) < 3:
        continue
    
    for film in watched.index[:3]:  # Her kullanıcının ilk 3 filmi
        actual = watched[film]
        
        # Benzer kullanıcılardan tahmin
        user_similarities = user_sim_df[user].drop(user).sort_values(ascending=False)
        top_similar = user_similarities.head(5).index
        
        weighted_sum = 0
        sim_sum = 0
        for sim_user in top_similar:
            rating = ratings_df.loc[sim_user, film]
            if not np.isnan(rating):
                sim = user_sim_df.loc[user, sim_user]
                weighted_sum += sim * rating
                sim_sum += sim
        
        if sim_sum > 0:
            predicted = weighted_sum / sim_sum
            error = abs(actual - predicted)
            errors.append(error)

if errors:
    mae = np.mean(errors)
    print(f"Ortalama Mutlak Hata (MAE): {mae:.2f}")
    print(f"Değerlendirilen tahmin sayısı: {len(errors)}")
else:
    print("Yeterli veri bulunamadı.")

## 8. Yaklaşımların Karşılaştırması

| Yaklaşım | Avantajları | Dezavantajları |
|----------|------------|----------------|
| **İçerik Tabanlı** | Soğuk başlangıç problemi yok (yeni filmler için), şeffaf öneriler | Çeşitlilik az, kullanıcı profili gerekli |
| **Kullanıcı Tabanlı İşbirlikçi** | Keşif potansiyeli yüksek, örtük tercihler yakalanır | Soğuk başlangıç, seyreklik, ölçeklenebilirlik |
| **Öğe Tabanlı İşbirlikçi** | Daha kararlı, ölçeklenebilir | Soğuk başlangıç, popülerlik eğilimi |
| **Hibrit** | En iyi sonuçlar, eksiklikleri tamamlar | Karmaşıklık, hesaplama maliyeti |

## Özet

Bu derste öğrendiklerimiz:
- **İçerik tabanlı filtreleme**: Film özelliklerine göre benzerlik hesaplar
- **İşbirlikçi filtreleme**: Kullanıcı davranışlarına göre öneri yapar
- **Kosinüs benzerliği** hem film hem kullanıcı benzerliği için kullanılabilir
- Gerçek dünyada **hibrit yaklaşımlar** (Netflix, Spotify gibi) en iyi sonuçları verir

### Alıştırma
1. Farklı bir benzerlik ölçüsü (Pearson korelasyonu) ile sonuçları karşılaştırın
2. Film sayısını 100'e çıkarıp sistemi test edin
3. Tür bazlı kişisel profil oluşturup içerik tabanlı öneri yapın

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://akademikyz.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>